# Addestramento SFCN - Classificazione Binaria (Dataset OpenNeuro FCD)
Questo notebook adatta l'architettura SFCN per un task di **classificazione binaria** (es. HC vs FCD) utilizzando il dataset OpenNeuro.

**Caratteristiche Principali:**
1. **Architettura Ridotta**: Utilizza il trick dei canali ridotti `[28, 58, ...]` proposto dagli autori originali per limitare l'overfitting nei task binari. L'addestramento avviene **from scratch**.
2. **5-Fold Cross Validation**: Applicata sull'80% del dataset.
3. **Valutazione Ensemble**: Il 20% di Test Set finale viene valutato confrontando il miglior modello singolo contro l'ensemble di tutte le 5 fold.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('./SFCN')

In [ ]:
import os
import json
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import seaborn as sns

from dp_model.model_files.sfcn import SFCN

PLOTS_DIR = '/kaggle/working/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)
MODELS_DIR = '/kaggle/working/models'
os.makedirs(MODELS_DIR, exist_ok=True)

## 1. Configurazione Parametri

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/datasets/collab4444/corrected_FCD_subject/corrected_FCD_subject" # Aggiorna col path corretto
MODALITY = "T1w"

# --- IMPOSTAZIONI K-FOLD E TRAINING ---
K_FOLDS = 5
OUTPUT_DIM = 2  # Classificazione Binaria (HC vs FCD)
EPOCHS = 150
BATCH_SIZE = 4
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 20

# TRICK AUTORI: Canali ridotti per task binari
CUSTOM_CHANNELS = [28, 58, 128, 256, 256, 64]


## 2. Dataloader per OpenNeuro (Estrazione Label da JSON)

In [ ]:
class OpenNeuroFCDDataset(Dataset):
    def __init__(self, data_dir, modality="T1w", is_train=False):
        self.data_dir = data_dir
        self.is_train = is_train
        self.samples = []
        
        if not os.path.exists(data_dir):
            print(f"ATTENZIONE: Directory {data_dir} non trovata.")
            return
            
        subjects = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)) and d.startswith("sub-")]
        
        raw_samples = []
        unique_groups = set()
        
        for sub in subjects:
            nii_path = os.path.join(data_dir, sub, f"{sub}_{modality}_MNI152_1mm.nii")
            if not os.path.exists(nii_path):
                nii_path = nii_path + ".gz"
                if not os.path.exists(nii_path):
                    if modality.lower() == 't1w':
                        nii_path_alt = os.path.join(data_dir, sub, f"{sub}_T1_MNI152_1mm.nii.gz")
                        if os.path.exists(nii_path_alt):
                            nii_path = nii_path_alt
                        else:
                            continue
                    else:
                        continue
                        
            json_path = os.path.join(data_dir, sub, f"{sub}_participant_info.json")
            if not os.path.exists(json_path):
                continue
                
            with open(json_path, 'r') as f:
                info = json.load(f)
                
            try:
                group = info['participant_info']['group'].lower()
                unique_groups.add(group)
                raw_samples.append({'nii_path': nii_path, 'group': group})
            except KeyError:
                continue
                
        unique_groups = sorted(list(unique_groups))
        self.class_map = {g: i for i, g in enumerate(unique_groups)}
        print(f"Mappatura Classi: {self.class_map}")
        
        for s in raw_samples:
            self.samples.append({
                'nii_path': s['nii_path'],
                'label': self.class_map[s['group']]
            })
            
        print(f"[{'TRAIN' if is_train else 'VAL/TEST'}] Caricati {len(self.samples)} pazienti validi.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        if self.is_train:
            dx = np.random.randint(-3, 4)
            dy = np.random.randint(-3, 4)
            dz = np.random.randint(-3, 4)
        else:
            dx, dy, dz = 0, 0, 0
            
        x_c = int((in_sp[0] - out_sp[0]) / 2) + dx
        y_c = int((in_sp[1] - out_sp[1]) / 2) + dy
        z_c = int((in_sp[2] - out_sp[2]) / 2) + dz
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        
        if self.is_train and np.random.rand() > 0.5:
            data = np.flip(data, axis=0).copy()
            
        data = np.expand_dims(data, axis=0)
        tensor_data = torch.from_numpy(data)
        label = torch.tensor(sample['label'], dtype=torch.long)
        
        return tensor_data, label

## 3. Split Principale (80% Train Pool, 20% Test Set)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in uso: {device}")

dummy_dataset = OpenNeuroFCDDataset(KAGGLE_DATA_DIR, modality=MODALITY, is_train=False)
dataset_size = len(dummy_dataset)

if dataset_size > 0:
    all_labels = [sample['label'] for sample in dummy_dataset.samples]
    all_indices = np.arange(dataset_size)
    
    train_pool_idx, test_idx, train_pool_labels, _ = train_test_split(
        all_indices, all_labels, 
        test_size=0.20, random_state=42, stratify=all_labels
    )
    
    print(f"\nSuddivisione Globale Completata:")
    print(f"- Pool K-Fold Training (80%): {len(train_pool_idx)} pazienti")
    print(f"- Test Set Finale (20%): {len(test_idx)} pazienti")
    
    test_dataset = OpenNeuroFCDDataset(KAGGLE_DATA_DIR, modality=MODALITY, is_train=False)
    test_dataset = torch.utils.data.Subset(test_dataset, test_idx)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)
else:
    print("ERRORE: Nessun file trovato.")

## 4. K-Fold CV & Addestramento Binario (From Scratch)

In [ ]:
def train_binary_model(model, train_loader, val_loader, device):
    criterion = nn.NLLLoss()
    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = optim.Adam(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.3)
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)[0].reshape(inputs.size(0), -1) # SFCN restituisce una lista [x]
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
        scheduler.step()
        
        model.eval()
        val_loss = 0.0
        correct = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)[0].reshape(inputs.size(0), -1)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                
                preds = torch.argmax(torch.exp(outputs), dim=1)
                correct += (preds == labels).sum().item()
                
        epoch_val_loss = val_loss / len(val_loader.dataset)
        
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            epochs_no_improve = 0
            best_model_state = model.state_dict()
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                break
                
    model.load_state_dict(best_model_state)
    return model, best_val_loss

best_models_paths = []
fold_val_losses = []

if dataset_size > 0:
    print("\n==============================================")
    print(f" INIZIO {K_FOLDS}-FOLD CV (FROM SCRATCH, CHANNELS REDUCED)")
    print("==============================================")
    
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    
    for fold, (train_inner_idx, val_inner_idx) in enumerate(skf.split(train_pool_idx, train_pool_labels)):
        print(f"\n--- Esecuzione FOLD {fold+1}/{K_FOLDS} ---")
        
        fold_train_idx = [train_pool_idx[i] for i in train_inner_idx]
        fold_val_idx = [train_pool_idx[i] for i in val_inner_idx]
        
        fold_train_dataset = OpenNeuroFCDDataset(KAGGLE_DATA_DIR, modality=MODALITY, is_train=True)
        fold_val_dataset = OpenNeuroFCDDataset(KAGGLE_DATA_DIR, modality=MODALITY, is_train=False)
        
        fold_train_dataset = torch.utils.data.Subset(fold_train_dataset, fold_train_idx)
        fold_val_dataset = torch.utils.data.Subset(fold_val_dataset, fold_val_idx)
        
        # Sampler Bilanciamento
        fold_train_labels = [all_labels[i] for i in fold_train_idx]
        class_counts = np.bincount(fold_train_labels)
        weights = [1.0 / class_counts[c] if class_counts[c] > 0 else 0 for c in fold_train_labels]
        sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)
        
        fold_train_loader = DataLoader(fold_train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
        fold_val_loader = DataLoader(fold_val_dataset, batch_size=1, shuffle=False, num_workers=2)
        
        # Inizializzazione modello from scratch con trick canali ridotti
        model = SFCN(output_dim=OUTPUT_DIM, channel_number=CUSTOM_CHANNELS)
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        model = model.to(device)
        
        trained_model, best_val_loss = train_binary_model(model, fold_train_loader, fold_val_loader, device)
        fold_val_losses.append(best_val_loss)
        
        model_save_path = os.path.join(MODELS_DIR, f"sfcn_openneuro_fold_{fold+1}.pth")
        if isinstance(trained_model, nn.DataParallel):
            torch.save(trained_model.module.state_dict(), model_save_path)
        else:
            torch.save(trained_model.state_dict(), model_save_path)
        
        best_models_paths.append(model_save_path)
        print(f">>> FOLD {fold+1} COMPLETATA | NLL Loss: {best_val_loss:.4f} | Salvato in: {model_save_path}")

## 5. Test Set Finale: Confronto Best Model vs Ensemble

In [ ]:
if dataset_size > 0:
    print("\n==============================================")
    print(" VALUTAZIONE TEST SET: BEST MODEL vs ENSEMBLE")
    print("==============================================")
    
    # --- A. Caricamento Modelli ---
    best_fold_idx = np.argmin(fold_val_losses)
    print(f"Miglior Fold Singola: {best_fold_idx+1} (Val Loss: {fold_val_losses[best_fold_idx]:.4f})")
    
    ensemble_models = []
    for path in best_models_paths:
        m = SFCN(output_dim=OUTPUT_DIM, channel_number=CUSTOM_CHANNELS)
        m.load_state_dict(torch.load(path, map_location=device))
        m = m.to(device)
        m.eval()
        ensemble_models.append(m)
        
    best_model = ensemble_models[best_fold_idx]
    
    # --- B. Inferenza --- 
    test_true_labels = []
    
    single_probs_1 = []
    single_preds = []
    
    ensemble_probs_1 = []
    ensemble_preds = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            test_true_labels.append(labels.item())
            
            # Best Single Model
            out_single = best_model(inputs)[0].reshape(inputs.size(0), -1)
            prob_single = torch.exp(out_single) # Converti da log_softmax a probabilitÃ 
            single_probs_1.append(prob_single[0, 1].item())
            single_preds.append(torch.argmax(prob_single, dim=1).item())
            
            # 5-Ensemble
            ens_p = []
            for m in ensemble_models:
                out_m = m(inputs)[0].reshape(inputs.size(0), -1)
                ens_p.append(torch.exp(out_m).cpu().numpy())
            mean_prob = np.mean(ens_p, axis=0)
            ensemble_probs_1.append(mean_prob[0, 1])
            ensemble_preds.append(np.argmax(mean_prob[0]))
            
    # --- C. Calcolo Metriche ---
    def evaluate(y_true, y_pred, y_prob, name):
        acc = accuracy_score(y_true, y_pred)
        try:
            auc = roc_auc_score(y_true, y_prob)
        except:
            auc = float('nan')
        print(f"\n>>> {name.upper()}")
        print(f"Accuracy : {acc:.4f}")
        print(f"ROC-AUC  : {auc:.4f}")
        return acc, auc
        
    acc_s, auc_s = evaluate(test_true_labels, single_preds, single_probs_1, "Best Model")
    acc_e, auc_e = evaluate(test_true_labels, ensemble_preds, ensemble_probs_1, "5-Ensemble")
    
    class_names = list(dummy_dataset.class_map.keys())
    
    # --- D. Grafici ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Bar Chart Confronto
    labels = ['Accuracy', 'ROC-AUC']
    single_scores = [acc_s, auc_s]
    ensemble_scores = [acc_e, auc_e]
    
    x = np.arange(len(labels))
    width = 0.35
    
    rects1 = ax1.bar(x - width/2, single_scores, width, label='Best Single Model', color='lightcoral', edgecolor='black')
    rects2 = ax1.bar(x + width/2, ensemble_scores, width, label='5-Ensemble', color='mediumseagreen', edgecolor='black')
    
    ax1.set_ylabel('Score', fontsize=12)
    ax1.set_title('Confronto Prestazioni sul Test Set', fontsize=14, fontweight='bold', pad=15)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, fontsize=12)
    ax1.set_ylim(0, 1.1)
    ax1.legend()
    
    for rect in rects1 + rects2:
        height = rect.get_height()
        if not np.isnan(height):
            ax1.annotate(f'{height:.3f}', xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Confusion Matrix (Ensemble)
    cm = confusion_matrix(test_true_labels, ensemble_preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax2)
    ax2.set_title("Confusion Matrix - 5-Ensemble", fontweight='bold', pad=15)
    ax2.set_xlabel("Predizione")
    ax2.set_ylabel("Reale")
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'openneuro_ensemble_comparison.png'), dpi=300)
    plt.show()